# Fundas_System -- Cataract Specialist Training v2 (luong rieng, plans/plan_cataract.txt)

FUNDAS_SYSTEM -- TRAIN CATARACT (ANH BAN TIEN/NGOAI NHAN CAU) TREN GOOGLE COLAB
====================================================
Boi canh: plans/plan_cataract.txt -- day la LUONG DAU TIEN trong he thong
KHONG dung anh fundus (xem chi tiet ly do/co hoi trong file .txt). Notebook
nay KHONG dung inference/preprocessing.py::load_and_crop (thiet ke rieng
cho anh fundus hinh tron co vien den) -- doc thang RGB, khong crop-theo-
nen-den.

Dataset: "Image Dataset on Eye Diseases Classification (Uveitis,
Conjunctivitis, Cataract, Eyelid)" (Kaggle: orvile/eye-diseases-
classification, Mendeley doi:10.17632/n9zp473wfw.1). Chi dung 2/5 thu muc:
Cataract/ va Normal/. Anh "thu thap tu Google search" -- ban quyen tung
anh khong ro rang, nguoi dung DA CHAP THUAN rui ro nay cho du an ca nhan.

===================== V2: SUA LOI PHAT HIEN TU LAN CHAY V1 =====================
Lan chay dau (v1, 2026-09-02) dung "ep tran do phan giai PIXEL" de chong
hoc-tat, nhung KET QUA THAT tu Colab cho thay GIA DINH BAN DAU SAI CHIEU:

    Sharpness (Laplacian variance) TRUOC xu ly (do THAT tren Colab):
      Cataract (187px trung vi):  227.2
      Normal   (56px trung vi):  1011.6   <-- CAO HON Cataract 4.5 LAN

Gia dinh ban dau (v1) la "anh pixel to (Cataract) se sac net hon, can lam
mo xuong cho khop Normal" -- SAI: anh thumbnail nho (Normal) lai co CHI SO
SAC NET (Laplacian variance) CAO HON HANG PHAN GIAI PIXEL LON (Cataract),
nhieu kha nang do thumbnail nho bi nen/sharpen qua tay luc tao (pho bien
voi anh web nho), "nhieu canh" nhieu hon du it chi tiet that.

Hau qua: ham "ep tran theo pixel" cua v1 lam mo NHAM Cataract (vi pixel to
hon nguong) trong khi De NGUYEN Normal (vi pixel nho hon nguong) -- khoang
cach sharpness KHONG dong lai ma con NOI RONG tu 4.5 lan len 23.6 lan (sau
xu ly: Cataract 36.7 vs Normal 865.8). Ket qua that: model dat val_f1=1.000
NGAY EPOCH 1, giu nguyen 25/25 epoch -- xac nhan day la hoc-tat qua chi so
sac net, khong phai hoc dau hieu duc thuy tinh the that.

SUA O V2: can bang truc tiep theo CHI SO SHARPNESS THAT (Laplacian
variance), KHONG con dung do phan giai pixel lam proxy nua:
  (1) Do sharpness (Laplacian variance) TRUOC xu ly cho ca 2 lop (nhu v1
      da lam, giu nguyen buoc nay -- day la buoc phat hien ra van de).
  (2) Xac dinh LOP SAC NET HON theo so do THAT (KHONG gia dinh truoc lop
      nao) -- co the la Cataract HOAC Normal tuy lan chay/dataset.
  (3) Hieu chinh 1 muc lam mo Gaussian (sigma) bang thu-dan (thu tang dan
      sigma tren 1 mau anh cua lop sac net hon, do lai sharpness sau moi
      lan thu) cho toi khi sharpness trung binh cua lop do giam ve GAN
      BANG lop kia -- ap dung CHINH XAC muc mo nay cho MOI anh cua lop sac
      net hon (khong dieu kien theo tung anh, tranh lap lai loi "dau vet
      xu ly" cua v1).
  (4) Do lai sharpness SAU xu ly, in ra so sanh truoc/sau nhu v1 -- neu
      van con chenh lech lon, PHAI DUNG LAI, khong duoc train tiep.

Output (2 file tai ve tu dong): model_best.pth + cataract_test_samples.zip
(giong v1).

CHUAN BI: giong v1 (mount Drive co kaggle.json, chon Runtime GPU, chay tuan
tu tu tren xuong). Cell 6 (xem anh mau sau xu ly) VAN la buoc BAT BUOC dung
lai kiem tra bang mat.

## CELL 1: Mount Drive + cau hinh Kaggle API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials ready.")

## CELL 2: Cai dependency + tai dataset

In [ ]:
!pip install -q kaggle timm

import kaggle
kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    'orvile/eye-diseases-classification',
    path='/content/cataract_raw',
    unzip=True,
    quiet=False,
)
print("Dataset downloaded.")

## CELL 3: Imports + tien xu ly RIENG cho anh ban tien (KHONG load_and_crop)

Anh ban tien la anh chu nhat binh thuong -- KHONG dung inference/
preprocessing.py::load_and_crop o day (ham do danh cho anh fundus hinh
tron co vien den).

In [ ]:
import hashlib
import json
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import timm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

RAW_ROOT = Path(
    '/content/cataract_raw/'
    'Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) '
    'with Symptoms and SMOTE Validation'
)

BACKBONE = 'seresnext50_32x4d'
IMAGE_SIZE = 224
DROPOUT = 0.3
VAL_FRACTION = 0.15
SEED = 0
EPOCHS = 25
BATCH_SIZE = 32
LR = 3e-4
N_TEST_SAMPLES = 10                # 5 cataract + 5 normal, xuat o Cell 10
CANDIDATE_BLUR_SIGMAS = [0, 0.5, 1, 1.5, 2, 3, 4, 6, 8, 10, 14, 18, 24]  # muc 9.7-v2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def load_rgb(path) -> np.ndarray:
    """Doc anh ban tien THANG, KHONG crop-theo-nen-den."""
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f'Could not read image: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def laplacian_variance(rgb: np.ndarray) -> float:
    """Thuoc do do sac net/chi tiet pho bien -- dung de PHAT HIEN va SUA
    confound do sac net giua 2 lop (plan_cataract.txt muc 9.8, v2)."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def apply_gaussian_blur(rgb: np.ndarray, sigma: float) -> np.ndarray:
    """Lam mo Gaussian voi do manh sigma -- sigma=0 tra ve nguyen anh."""
    if sigma <= 0:
        return rgb
    k = max(3, int(round(sigma * 3)) | 1)  # kernel size le, ti le voi sigma
    return cv2.GaussianBlur(rgb, (k, k), sigma)


_MEAN, _STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
try:
    cfg = timm.get_pretrained_cfg(BACKBONE)
    if cfg is not None and cfg.mean and cfg.std:
        _MEAN, _STD = cfg.mean, cfg.std
except Exception:
    pass

_RESIZE_NORM = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_MEAN, std=_STD),
])


def preprocess(rgb: np.ndarray) -> torch.Tensor:
    return _RESIZE_NORM(Image.fromarray(rgb))

## CELL 4: Xay manifest -- loc trung lap (hash) + do sharpness GOC + HIEU CHINH muc lam mo

KHONG con dung do phan giai pixel de quyet dinh xu ly (day la loi cua v1)
-- CHI dung sharpness (Laplacian variance) THAT de xac dinh lop nao can
lam mo va lam mo BAO NHIEU.

In [ ]:
def file_hash(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()


def load_class(folder: str, label: int):
    rows = []
    seen_hashes = set()
    n_total, n_dupe = 0, 0
    for p in sorted((RAW_ROOT / folder).iterdir()):
        if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        n_total += 1
        h = file_hash(p)
        if h in seen_hashes:
            n_dupe += 1
            continue
        seen_hashes.add(h)
        rgb = load_rgb(p)
        native_short_side = min(rgb.shape[0], rgb.shape[1])
        sharpness = laplacian_variance(rgb)
        rows.append({'path': p, 'label': label, 'native_short_side': native_short_side, 'sharpness_before': sharpness})
    print(f'  {folder}: {n_total} files -> {len(rows)} unique after hash-dedup ({n_dupe} duplicates dropped)')
    return rows


print('Loading + deduplicating + measuring native resolution + sharpness (reads every image once):')
cataract_rows = load_class('Cataract', 1)
normal_rows = load_class('Normal', 0)
all_rows = cataract_rows + normal_rows
print(f'\nTotal unique images: {len(all_rows)}  (cataract={len(cataract_rows)}, normal={len(normal_rows)})')

median_cataract_res = float(np.median([r['native_short_side'] for r in cataract_rows]))
median_normal_res = float(np.median([r['native_short_side'] for r in normal_rows]))
print(f'Median native short side (INFO ONLY, not used for equalization anymore -- v1 bug) -- '
      f'cataract: {median_cataract_res:.0f}px  normal: {median_normal_res:.0f}px')

sharpness_cataract_before = float(np.mean([r['sharpness_before'] for r in cataract_rows]))
sharpness_normal_before = float(np.mean([r['sharpness_before'] for r in normal_rows]))
print(f"\nSharpness (Laplacian variance) BEFORE equalization -- "
      f"cataract: {sharpness_cataract_before:.1f}  normal: {sharpness_normal_before:.1f}")

# --- Xac dinh lop SAC NET HON THEO SO DO THAT (khong gia dinh truoc) ---
if sharpness_cataract_before > sharpness_normal_before:
    SHARPER_LABEL, sharper_name = 1, 'cataract'
    target_sharpness = sharpness_normal_before
else:
    SHARPER_LABEL, sharper_name = 0, 'normal'
    target_sharpness = sharpness_cataract_before
print(f'\nSharper class: {sharper_name}  ->  se lam mo lop nay xuong muc ~{target_sharpness:.1f}')

# --- Hieu chinh sigma bang thu-dan tren 1 mau cua lop sac net hon ---
sharper_rows = [r for r in all_rows if r['label'] == SHARPER_LABEL]
calib_sample = random.Random(SEED).sample(sharper_rows, min(60, len(sharper_rows)))
calib_images = [load_rgb(r['path']) for r in calib_sample]

print('\nHieu chinh BLUR_SIGMA (thu tang dan cho den khi sharpness ve gan muc muc tieu):')
BLUR_SIGMA = CANDIDATE_BLUR_SIGMAS[-1]
for sigma in CANDIDATE_BLUR_SIGMAS:
    avg_var = float(np.mean([laplacian_variance(apply_gaussian_blur(im, sigma)) for im in calib_images]))
    print(f'  sigma={sigma:>4.1f}  ->  avg sharpness = {avg_var:.1f}')
    if avg_var <= target_sharpness:
        BLUR_SIGMA = sigma
        break

print(f'\nChosen BLUR_SIGMA = {BLUR_SIGMA}, applied to ALL images of label={SHARPER_LABEL} ({sharper_name}).')

assert len(all_rows) > 500, 'Expected roughly 1000+ unique images total -- check RAW_ROOT path above if this fails.'

## CELL 5: Split train/val (stratify theo label)

In [ ]:
def stratified_split(rows, val_fraction, seed):
    rng = random.Random(seed)
    groups = {}
    for row in rows:
        groups.setdefault(row['label'], []).append(row)

    train_rows, val_rows = [], []
    for label, group in groups.items():
        group = group[:]
        rng.shuffle(group)
        n_val = max(1, round(len(group) * val_fraction))
        val_rows.extend(group[:n_val])
        train_rows.extend(group[n_val:])
    return train_rows, val_rows


train_rows, val_rows = stratified_split(all_rows, VAL_FRACTION, SEED)
print(f'Train: {len(train_rows)}  Val: {len(val_rows)}')
print('Val per label (1=cataract):', {l: sum(1 for r in val_rows if r['label'] == l) for l in (0, 1)})

## CELL 6: Dataset/DataLoader + XEM TRUOC anh SAU lam mo -- BAT BUOC DUNG LAI KIEM TRA

Ap dung apply_gaussian_blur(rgb, BLUR_SIGMA) CHO MOI ANH CO label ==
SHARPER_LABEL (khong dieu kien theo tung anh -- tranh lap lai loi "dau vet
xu ly khong dong deu" cua v1). In lai sharpness SAU xu ly de xac nhan
khoang cach da dong lai THAT, roi xem 10 anh mau bang mat.

In [ ]:
class CataractDataset(Dataset):
    def __init__(self, rows, blur_sigma: float, sharper_label: int, augment: bool):
        self.rows = rows
        self.blur_sigma = blur_sigma
        self.sharper_label = sharper_label
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def _base_transform(self, rgb, label):
        if label == self.sharper_label:
            rgb = apply_gaussian_blur(rgb, self.blur_sigma)
        return rgb

    def __getitem__(self, idx):
        row = self.rows[idx]
        rgb = load_rgb(row['path'])
        rgb = self._base_transform(rgb, row['label'])

        if self.augment:
            if np.random.rand() < 0.5:
                rgb = np.ascontiguousarray(rgb[:, ::-1])
            # Lop bao hiem them (plan_cataract.txt muc 9.5): mo NGAU NHIEN
            # THEM (nho) CHO CA HAI LOP, chi luc train -- khien model khong
            # the dua vao 1 muc do net co dinh chinh xac tuyet doi.
            if np.random.rand() < 0.3:
                rgb = apply_gaussian_blur(rgb, float(np.random.uniform(0.5, 2.0)))

        return preprocess(rgb), torch.tensor(row['label'], dtype=torch.float32)


train_labels = np.array([r['label'] for r in train_rows])
n_pos, n_neg = train_labels.sum(), len(train_labels) - train_labels.sum()
pos_weight = torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32)
print(f'Train pos(cataract)={int(n_pos)} neg(normal)={int(n_neg)} pos_weight={pos_weight.item():.3f}')

train_loader = DataLoader(CataractDataset(train_rows, BLUR_SIGMA, SHARPER_LABEL, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(CataractDataset(val_rows, BLUR_SIGMA, SHARPER_LABEL, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- Kiem chung dinh luong SAU xu ly (BAT BUOC doc so nay truoc khi train) ---
check_cataract = [apply_gaussian_blur(load_rgb(r['path']), BLUR_SIGMA if r['label'] == SHARPER_LABEL else 0) for r in cataract_rows[:60]]
check_normal = [apply_gaussian_blur(load_rgb(r['path']), BLUR_SIGMA if r['label'] == SHARPER_LABEL else 0) for r in normal_rows[:60]]
sharpness_cataract_after = float(np.mean([laplacian_variance(im) for im in check_cataract]))
sharpness_normal_after = float(np.mean([laplacian_variance(im) for im in check_normal]))
print(f'\nSharpness AFTER equalization (60-image sample/class) -- '
      f'cataract: {sharpness_cataract_after:.1f}  normal: {sharpness_normal_after:.1f}')
gap_before = max(sharpness_cataract_before, sharpness_normal_before) / max(min(sharpness_cataract_before, sharpness_normal_before), 1e-6)
gap_after = max(sharpness_cataract_after, sharpness_normal_after) / max(min(sharpness_cataract_after, sharpness_normal_after), 1e-6)
print(f'Sharpness ratio BEFORE: {gap_before:.2f}x   AFTER: {gap_after:.2f}x')
print('NEU ty le AFTER van con lon (vd > 2x) -- DUNG LAI, KHONG chay Cell 7,')
print('bao lai ket qua de dieu chinh (vd mo rong CANDIDATE_BLUR_SIGMAS o Cell 3).')

# --- Xem truoc 10 anh SAU xu ly (BAT BUOC kiem tra bang mat) ---
import matplotlib.pyplot as plt

preview_rows = random.Random(SEED).sample(all_rows, min(10, len(all_rows)))
fig, axs = plt.subplots(2, 5, figsize=(22, 10))
for ax, row in zip(axs.flat, preview_rows):
    rgb = load_rgb(row['path'])
    if row['label'] == SHARPER_LABEL:
        rgb = apply_gaussian_blur(rgb, BLUR_SIGMA)
    ax.imshow(rgb)
    tag = 'cataract' if row['label'] == 1 else 'normal'
    ax.set_title(f"{tag}\n{row['path'].name}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('\nKiem tra bang mat: voi do net nay, ban CON phan biet duoc dong tu')
print('duc/mo (cataract) voi dong tu trong, binh thuong (normal) khong?')
print('Neu KHONG -- dung lai, bao ket qua Cell 4+6 (cac so sharpness va ty le)')
print('de dieu chinh lai, KHONG chay Cell 7 tro di.')

## CELL 7: Model -- phan loai nhi phan (1 logit, sigmoid = P(cataract))

Cung mau voi inference/glaucoma_model.py va inference/amd_model.py.

In [ ]:
class CataractModel(nn.Module):
    def __init__(self, backbone_name, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(-1)  # raw logit


model = CataractModel(BACKBONE, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

## CELL 8: Vong lap train -- chon model tot nhat theo val F1 (lop cataract)

CANH BAO (rut kinh nghiem tu chinh lan chay v1 cua notebook nay VA tu AMD):
neu val_f1/val_acc = 1.000 NGAY EPOCH 1 va giu nguyen nhieu epoch, DUNG
VOI TIN -- day gan chac la dau hieu con confound chua duoc xu ly het, quay
lai doc ky so lieu sharpness o Cell 4/6 truoc khi tin ket qua.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        probs = torch.sigmoid(model(x)).cpu()
        all_probs.append(probs)
        all_labels.append(y)
    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds = (probs >= 0.5).astype(int)

    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (preds == labels).mean()
    return {'accuracy': float(accuracy), 'precision': precision, 'recall': recall, 'f1': f1}


best_f1 = -1.0
best_state = None
best_metrics = {}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_rows)
    metrics = evaluate(model, val_loader)
    print(f"epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss:.4f}  "
          f"val_acc={metrics['accuracy']:.3f}  val_precision={metrics['precision']:.3f}  "
          f"val_recall={metrics['recall']:.3f}  val_f1={metrics['f1']:.3f}")

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_metrics = {'val_accuracy': metrics['accuracy'], 'val_precision': metrics['precision'],
                         'val_recall': metrics['recall'], 'val_f1': metrics['f1'], 'epoch': epoch}

print(f"\nBest: val_f1={best_f1:.3f}  (epoch {best_metrics.get('epoch')})")
print('\nDOC LAI toan bo log epoch o tren truoc khi tin con so nay -- xem canh bao dau Cell nay.')

## CELL 9: Luu checkpoint + TU DONG tai ve may

In [ ]:
CHECKPOINT_NAME = 'model_best.pth'
torch.save(
    {
        'model_state_dict': best_state,
        'labels': ['normal', 'cataract'],
        'backbone': BACKBONE,
        'dropout': DROPOUT,
        'sharper_label': SHARPER_LABEL,
        'blur_sigma': BLUR_SIGMA,
        **best_metrics,
    },
    CHECKPOINT_NAME,
)

shutil.copy(CHECKPOINT_NAME, '/content/drive/MyDrive/' + CHECKPOINT_NAME)
print('Saved to Google Drive:', '/content/drive/MyDrive/' + CHECKPOINT_NAME)

from google.colab import files
files.download(CHECKPOINT_NAME)

print(json.dumps({
    'task': 'Cataract Specialist Screening (anterior/external eye photo, plans/plan_cataract.txt, v2)',
    'dataset_source': 'https://www.kaggle.com/datasets/orvile/eye-diseases-classification (Mendeley doi:10.17632/n9zp473wfw.1)',
    'backbone': BACKBONE,
    'image_size': IMAGE_SIZE,
    'dropout': DROPOUT,
    'labels': ['normal', 'cataract'],
    'checkpoint': CHECKPOINT_NAME,
    'is_experimental': True,
    'not_fundus_based': True,
    'requires_eye_roi_crop': True,
    'sharper_label': SHARPER_LABEL,
    'blur_sigma_applied_to_sharper_class': BLUR_SIGMA,
    'sharpness_before_equalization': {'cataract': sharpness_cataract_before, 'normal': sharpness_normal_before},
    'sharpness_after_equalization': {'cataract': sharpness_cataract_after, 'normal': sharpness_normal_after},
    'known_caveats': [
        'v1 of this notebook (2026-09-02) equalized by NATIVE PIXEL RESOLUTION, which turned out to be the WRONG proxy -- the class with smaller native pixels (Normal, ~56px) actually had HIGHER measured sharpness (Laplacian variance ~1012) than the larger-pixel class (Cataract, ~187px, sharpness ~227), likely due to web-thumbnail sharpening/compression artifacts. v1 blurred the wrong class, widening the sharpness gap from 4.5x to 23.6x and producing a trivially-shortcuttable model (val_f1=1.000 from epoch 1). v2 (this run) equalizes directly on measured Laplacian variance instead of pixel resolution -- see sharpness_before/after above for whether it actually closed the gap this time.',
        'Source images were collected by the dataset author via Google Image Search, not a controlled clinical acquisition -- per-image copyright uncertain, accepted as a known risk for this personal, non-commercial project (plans/plan_cataract.txt section 8).',
        'This model expects an already-cropped close-up eye photo (same framing as training data) -- inference/cataract_engine.py (not built yet) must add an eye-ROI auto-crop step for real user uploads.',
    ],
    **best_metrics,
}, indent=4))

## CELL 10: Xuat 10 anh ngau nhien, TY LE 5 cataract / 5 normal (tu tap validation) de test local

In [ ]:
SAMPLE_DIR = Path('/content/cataract_test_samples')
if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)
SAMPLE_DIR.mkdir(parents=True)

rng = random.Random(SEED + 1)
cataract_val_rows = [r for r in val_rows if r['label'] == 1]
normal_val_rows = [r for r in val_rows if r['label'] == 0]

n_each = N_TEST_SAMPLES // 2
sample_rows = (
    rng.sample(cataract_val_rows, min(n_each, len(cataract_val_rows)))
    + rng.sample(normal_val_rows, min(n_each, len(normal_val_rows)))
)
rng.shuffle(sample_rows)

label_names = {0: 'normal', 1: 'cataract'}
manifest = []
for i, row in enumerate(sample_rows, start=1):
    dest_name = f"{i:02d}_{label_names[row['label']]}{row['path'].suffix}"
    shutil.copy(row['path'], SAMPLE_DIR / dest_name)
    manifest.append({'filename': dest_name, 'true_label': label_names[row['label']]})

n_cataract = sum(1 for m in manifest if m['true_label'] == 'cataract')
n_normal = sum(1 for m in manifest if m['true_label'] == 'normal')

with open(SAMPLE_DIR / 'labels.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'{len(sample_rows)} sample images copied to {SAMPLE_DIR} (cataract={n_cataract}, normal={n_normal}):')
for m in manifest:
    print(' ', m)

ZIP_PATH = '/content/cataract_test_samples.zip'
shutil.make_archive('/content/cataract_test_samples', 'zip', SAMPLE_DIR)

files.download(ZIP_PATH)
print(f'\nDownloaded: {ZIP_PATH}')

## SAU KHI TAI VE MAY LOCAL

1. Copy `model_best.pth` vao `D:\Fundas_System\models\cataract\model_best.pth`.
2. Copy JSON o Cell 9 vao `D:\Fundas_System\config\cataract\config.json`.
3. Giai nen `cataract_test_samples.zip` vao 1 thu muc cuc bo de kiem thu sau.
4. QUAN TRONG: PHAI doc lai log Cell 8 that ky (khong chi con so cuoi). Neu
   VAN thay val_f1=1.000 tu epoch 1 -- BAO LAI NGAY, DUNG copy checkpoint
   vao he thong, day van la dau hieu con confound khac chua duoc xu ly.
5. Buoc auto-crop vung mat (plan_cataract.txt muc 2.2(a)) van la viec lam
   SAU trong inference/cataract_engine.py, khong nam trong notebook nay.